# TRTR Dataset A - Diabetes

In [1]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
print('Libraries imported!!')

Libraries imported!!


In [2]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/UTILITY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)
#import functions for data labelling analisys
from utility_evaluation import DataPreProcessor
from utility_evaluation import train_evaluate_model

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read data

In [3]:
#read real dataset
train_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Train.csv')
categorical_columns = ['group']
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category')
train_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,2.009493,0.084640,-0.020160,0.107037,0.857051,-0.048580,0.141517,0.137274
1,Group0,1.647881,0.285048,0.750705,-0.365256,0.594206,-0.094862,0.201428,-0.240859
2,Group0,0.045838,-0.301003,-0.007641,0.189160,1.875668,-0.152879,0.003445,0.012916
3,Group0,-1.626382,0.438629,-0.352224,-0.147169,-2.445341,0.362086,-0.132354,-0.016382
4,Group0,-0.244556,-0.781058,0.191160,0.637441,-2.766078,-1.110363,0.746589,0.140332
...,...,...,...,...,...,...,...,...,...
2712,Group0,0.048656,0.340805,0.146978,-0.077990,1.468295,-0.105885,0.037067,-0.001841
2713,Group0,-3.007362,-0.110868,-0.547719,-0.061449,-3.286850,-0.079093,-0.126217,0.165070
2714,Group0,0.481546,-0.118290,-0.159216,-0.046665,0.013795,0.235801,-0.128855,0.206748
2715,Group0,0.199530,0.371273,0.268324,0.670192,0.275167,-0.382674,0.076358,0.294886


In [4]:
#read test data
test_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category')
test_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-3.053475,-0.177497,-0.010661,0.205118,-2.773102,-0.653440,0.319585,0.017457
1,Group0,-1.375254,0.761393,0.270692,0.140597,-2.157208,-0.297755,0.126143,0.118270
2,Group0,3.681521,1.147414,-0.119372,0.276912,1.452468,0.068735,0.043671,0.074436
3,Group0,0.383070,0.211771,0.374537,0.254718,5.352641,-0.611636,0.344336,0.044627
4,Group0,2.009235,0.765321,0.100557,0.058659,2.842860,0.980271,-0.014527,0.096506
...,...,...,...,...,...,...,...,...,...
674,Group0,1.185955,-0.248240,-0.258972,0.284599,0.816753,0.553623,-0.456024,0.157975
675,Group0,-4.898967,-0.576216,-0.150236,0.069086,-4.450551,-0.192307,0.034382,-0.174072
676,Group0,-3.339095,0.856460,-1.021265,-0.131611,0.639099,0.783467,-0.128578,-0.067689
677,Group0,-3.844234,0.083773,0.334898,-0.210940,-1.259774,0.726944,-0.299066,-0.123455


In [5]:
target = 'group'
#quick look at the breakdown of class values
print('Train data')
print(train_data.shape)
print(train_data.groupby(target).size())
print('#####################################')
print('Test data')
print(test_data.shape)
print(test_data.groupby(target).size())

Train data
(2717, 9)
group
Group0    2606
Group1     111
dtype: int64
#####################################
Test data
(679, 9)
group
Group0    645
Group1     34
dtype: int64


## 2. Pre-process training data

In [6]:
target = 'group'

# 先分离特征和目标
X = train_data.drop(columns=[target])
y = train_data[target]

# 然后基于 X 定义列
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# categories 需要根据你的实际分类列调整
categories = [np.array(range(2))] if categorical_columns else []

data_preprocessor = DataPreProcessor(categorical_columns, numerical_columns, categories)
x_train = data_preprocessor.preprocess_train_data(X)
y_train = y

x_train.shape, y_train.shape

((2717, 8), (2717,))

## 3. Preprocess test data

In [7]:
x_test = data_preprocessor.preprocess_test_data(test_data.loc[:, test_data.columns != target])
y_test = test_data.loc[:, target]
x_test.shape, y_test.shape

((679, 8), (679,))

## 4. Create a dataset to save the results

In [8]:
results = pd.DataFrame(columns = ['model','accuracy','precision','recall','f1'])
results

,model,accuracy,precision,recall,f1


## 4. Train and evaluate Random Forest Classifier

In [9]:
rf_results = train_evaluate_model('RF', x_train, y_train, x_test, y_test)
results = pd.concat([results, rf_results], ignore_index=True)
rf_results

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.1s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.0s finished


,model,accuracy,precision,recall,f1
0,RF,0.9558,0.9478,0.9558,0.9426


## 5. Train and Evaluate KNeighbors Classifier

In [10]:
knn_results = train_evaluate_model('KNN', x_train, y_train, x_test, y_test)
results = pd.concat([results, knn_results], ignore_index=True)
knn_results

,model,accuracy,precision,recall,f1
0,KNN,0.9485,0.9202,0.9485,0.9275


## 6. Train and evaluate Decision Tree Classifier

In [11]:
dt_results = train_evaluate_model('DT', x_train, y_train, x_test, y_test)
results = pd.concat([results, dt_results], ignore_index=True)
dt_results

,model,accuracy,precision,recall,f1
0,DT,0.9234,0.9234,0.9234,0.9234


## 7. Train and evaluate Support Vector Machines Classifier

In [12]:
svm_results = train_evaluate_model('SVM', x_train, y_train, x_test, y_test)
results = pd.concat([results, svm_results], ignore_index=True)
svm_results

[LibSVM]WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -562.956779, rho = 0.778622
nSV = 156, nBSV = 0
Total nSV = 156
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -706.155908, rho = 1.555257
nSV = 166, nBSV = 0
Total nSV = 166
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -737.122458, rho = 0.736143
nSV = 161, nBSV = 0
Total nSV = 161
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -804.430488, rho = 1.186184
nSV = 149, nBSV = 1
Total nSV = 149
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -652.910070, rho = 1.590597
nSV = 176, nBSV = 0
Total nSV = 176
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -189.818454, rho = -0.383538
nSV = 124, nBSV = 0
Total nSV = 124


,model,accuracy,precision,recall,f1
0,SVM,0.6627,0.9286,0.6627,0.7575


## 8. Train and evaluate Multilayer Perceptron Classifier

In [13]:
mlp_results = train_evaluate_model('MLP', x_train, y_train, x_test, y_test)
results = pd.concat([results, mlp_results], ignore_index=True)
mlp_results

Iteration 1, loss = 0.54521211
Iteration 2, loss = 0.27829981
Iteration 3, loss = 0.18840587
Iteration 4, loss = 0.16639010
Iteration 5, loss = 0.15065750
Iteration 6, loss = 0.14319027
Iteration 7, loss = 0.13687124
Iteration 8, loss = 0.13171297
Iteration 9, loss = 0.12695852
Iteration 10, loss = 0.12218133
Iteration 11, loss = 0.11909902
Iteration 12, loss = 0.11523355
Iteration 13, loss = 0.11238482
Iteration 14, loss = 0.11028170
Iteration 15, loss = 0.10794167
Iteration 16, loss = 0.10625930
Iteration 17, loss = 0.10442823
Iteration 18, loss = 0.10323876
Iteration 19, loss = 0.10257910
Iteration 20, loss = 0.10183838
Iteration 21, loss = 0.09974436
Iteration 22, loss = 0.09870070
Iteration 23, loss = 0.09803130
Iteration 24, loss = 0.09805747
Iteration 25, loss = 0.09730066
Iteration 26, loss = 0.09562086
Iteration 27, loss = 0.09368057
Iteration 28, loss = 0.09282467
Iteration 29, loss = 0.09794524
Iteration 30, loss = 0.09479793
Iteration 31, loss = 0.09198375
Iteration 32, los

,model,accuracy,precision,recall,f1
0,MLP,0.9426,0.9333,0.9426,0.9373


## 9. Save results file

In [14]:
results.to_csv('RESULTS/models_results_real.csv', index=False)
results

,model,accuracy,precision,recall,f1
0,RF,0.9558,0.9478,0.9558,0.9426
1,KNN,0.9485,0.9202,0.9485,0.9275
2,DT,0.9234,0.9234,0.9234,0.9234
3,SVM,0.6627,0.9286,0.6627,0.7575
4,MLP,0.9426,0.9333,0.9426,0.9373
